# Stage 2 — Checking the standardized table

`pipeline/standardize.py` turned the raw text data (3 column layouts) into **one typed table**: `data/interim/standardized.parquet`.
This notebook checks that nothing was lost and that the time-zone handling is right.

**Note:** every query sets DuckDB's time zone to UTC so the hours shown are UTC, not shifted to this computer's zone.

In [1]:
import duckdb
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
S = (ROOT / "data/interim/standardized.parquet").as_posix()
RAW = f"read_parquet('{(ROOT / 'data/interim/raw_parquet/*.parquet').as_posix()}', union_by_name = true)"

con = duckdb.connect()
con.sql("SET TimeZone = 'UTC'")
q = lambda sql: con.sql(sql).df()

## 1. Schema: column names and types

In [2]:
q(f"SELECT column_name, column_type FROM (DESCRIBE SELECT * FROM '{S}')")

,column_name,column_type
0,service_request_num,VARCHAR
1,sr_record_id,VARCHAR
2,sr_type,VARCHAR
3,method_received,VARCHAR
4,created_date,TIMESTAMP WITH TIME ZONE
5,status_date,TIMESTAMP WITH TIME ZONE
6,due_date,TIMESTAMP WITH TIME ZONE
7,close_date,TIMESTAMP WITH TIME ZONE
8,last_activity_date,TIMESTAMP WITH TIME ZONE
9,sr_status,VARCHAR


- Dates are `TIMESTAMP WITH TIME ZONE` and stored in UTC (Postgres `timestamptz` later).
- `zip_code` and `police_post` stay **text**: zip codes are labels, and some police posts list several values (`111,112`).
- `ts_suspect` marks the 2026 rows whose times were re-interpreted (see section 4).

## 2. Nothing lost: row count and unique IDs

In [3]:
q(f'''SELECT (SELECT count(*) FROM {RAW}) AS raw_rows,
             count(*) AS standardized_rows,
             count(DISTINCT service_request_num) AS distinct_service_request_num
      FROM '{S}' ''')

,raw_rows,standardized_rows,distinct_service_request_num
0,13019163,13019163,13019163


## 3. Nothing failed to parse
For each column: rows that had a value in the raw data but are empty after conversion. All should be **0**.

In [4]:
checks = {"created_date": "CreatedDate", "status_date": "StatusDate", "due_date": "DueDate",
          "close_date": "CloseDate", "last_activity_date": "LastActivityDate",
          "council_district": "CouncilDistrict", "latitude": "Latitude", "longitude": "Longitude"}
raw_counts = q("SELECT " + ", ".join(f"count(nullif(trim({r}),'')) AS {n}" for n, r in checks.items()) + f" FROM {RAW}")
new_counts = q("SELECT " + ", ".join(f"count({n}) AS {n}" for n in checks) + f" FROM '{S}'")
(raw_counts - new_counts).rename(index={0: "failed_to_parse"})

,created_date,status_date,due_date,close_date,last_activity_date,council_district,latitude,longitude
failed_to_parse,0,0,0,0,0,0,0,0


## 4. Time zones

Three groups of rows:
- **2010–2025 (and some 2026):** written as Baltimore local time with no zone → converted from `America/New_York` to UTC (daylight saving handled).
- **2026, `ts_suspect = false`:** really UTC → kept.
- **2026, `ts_suspect = true`:** record last touched on/after 2026-08-16, when the source started writing local time labeled `+00` → re-interpreted as local and converted.

**Check A: phone-call hour of day (UTC).** People call in business hours, 9am–5pm Baltimore = roughly **13–21 UTC**. If the conversion is right, all three groups look the same.

In [5]:
df = q(f'''SELECT CASE WHEN year(created_date) < 2026 THEN '2024-25'
                       WHEN NOT ts_suspect THEN '2026 (UTC as labeled)'
                       ELSE '2026 ts_suspect (corrected)' END AS grp,
                  hour(created_date) AS hour_utc, count(*) AS n
           FROM '{S}' WHERE method_received = 'Phone' AND year(created_date) >= 2024 GROUP BY 1, 2''')
p = df.pivot(index="grp", columns="hour_utc", values="n").fillna(0)
(100 * p.div(p.sum(axis=1), axis=0)).round(0).astype(int)

hour_utc,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
grp,,,,,,,,,,,,,,,,,,,,,
2024-25,3,3,2,0,0,0,0,0,0,0,...,9,9,9,9,8,8,7,6,5,4
2026 (UTC as labeled),3,3,4,0,0,0,0,0,0,0,...,9,9,9,9,8,8,7,7,5,4
2026 ts_suspect (corrected),3,4,1,1,0,0,0,0,0,0,...,8,8,8,8,9,9,7,6,5,4


**Check B: due times.** Due dates are usually set to local end-of-day, 11:59pm. In UTC that is **03:59** in summer (UTC−4) and **04:59** in winter (UTC−5).

In [6]:
q(f'''SELECT CASE WHEN year(created_date) < 2026 THEN '2024-25'
                 WHEN NOT ts_suspect THEN '2026 (UTC as labeled)'
                 ELSE '2026 ts_suspect (corrected)' END AS grp,
            strftime(due_date, '%H:%M') AS due_time_utc, count(*) AS n
     FROM '{S}' WHERE year(created_date) >= 2024 AND minute(due_date) = 59
     GROUP BY 1, 2 QUALIFY row_number() OVER (PARTITION BY grp ORDER BY n DESC) <= 3
     ORDER BY 1, 3 DESC''')

,grp,due_time_utc,n
0,2024-25,03:59,32930
1,2024-25,04:59,23590
2,2024-25,23:59,5670
3,2026 (UTC as labeled),04:59,16563
4,2026 (UTC as labeled),16:59,1107
5,2026 (UTC as labeled),15:59,1049
6,2026 ts_suspect (corrected),03:59,5418
7,2026 ts_suspect (corrected),08:59,2720
8,2026 ts_suspect (corrected),04:59,2718


In the corrected group, **03:59** dominates as expected. The **08:59** rows are most likely records the ~90% rule got wrong: they were really UTC (04:59), and got shifted 4 more hours. That is the known cost of option (c), and those rows are all marked `ts_suspect`.

In [7]:
q(f"SELECT year(created_date) AS yr, count(*) AS n, count(*) FILTER (WHERE ts_suspect) AS ts_suspect FROM '{S}' WHERE year(created_date) >= 2025 GROUP BY 1 ORDER BY 1")

,yr,n,ts_suspect
0,2025,931365,0
1,2026,975204,268305


## 5. Spot check: one record, raw vs standardized

In [8]:
srn = "24-00000297"
display(q(f"SELECT ServiceRequestNum, CreatedDate, DueDate, CloseDate FROM {RAW} WHERE ServiceRequestNum = '{srn}'"))
q(f"SELECT service_request_num, created_date::VARCHAR AS created_utc, due_date::VARCHAR AS due_utc, close_date::VARCHAR AS close_utc FROM '{S}' WHERE service_request_num = '{srn}'")

,ServiceRequestNum,CreatedDate,DueDate,CloseDate
0,24-00000297,1/1/2024 7:51:28 AM,2/10/2024 11:59:00 PM,2/10/2024 10:22:00 AM


,service_request_num,created_utc,due_utc,close_utc
0,24-00000297,2024-01-01 12:51:28+00,2024-02-11 04:59:00+00,2024-02-10 15:22:00+00


February is standard time (UTC−5), so `11:59 PM` local → `04:59` UTC the next day. ✓

## 6. Cleaned text: police districts

In [9]:
q(f"SELECT police_district, count(*) AS n FROM '{S}' GROUP BY 1 ORDER BY 2 DESC")

,police_district,n
0,NaN,2541292
1,Northeastern,1674513
2,Southeastern,1330530
3,Northern,1284159
4,Southwestern,1190701
5,Southern,1097760
6,Eastern,1085617
7,Northwestern,1044304
8,Western,1022995
9,Central,747279


One spelling per district now (`NORTHERN` and `Northern` merged). A handful of stray values (zip codes, `East`/`West`) were left as they are.